In [ ]:
import os
import pandas as pd
import xgboost as xgb
import bottleneck as bn

from utils import download_crypto_data
from datetime import date
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

In [ ]:
download_crypto_data(symbol="BTCUSDT", start=date(2024, 1, 1), end=date(2025, 12, 31), interval="1h", suppress_info=True)

In [ ]:
train_data = pd.read_csv("data/BTCUSDT-1h-2019-01-01_2023-12-31.csv")
test_data = pd.read_csv("data/BTCUSDT-1h-2024-01-01_2025-12-31.csv")
train_df = pd.DataFrame()
test_df = pd.DataFrame()

Feature Engineering

In [ ]:
train_data["RelativeDiff"] = (train_data["High"] - train_data["Low"]) / (train_data["High"] + train_data["Low"])
test_data["RelativeDiff"] = (test_data["High"] - test_data["Low"]) / (test_data["High"] + test_data["Low"])

windows = [10, 25, 100]
configs = ["Volume", "NumberOfTrades", "TakerBuyBaseVolume", "Close"]

for source_col in configs:
    train_rolling = {w: train_data[source_col].rolling(w) for w in windows}
    test_rolling = {w: test_data[source_col].rolling(w) for w in windows}

    for w in windows:
        train_df[f"FEATURE_{source_col}Skew{w}"] = train_rolling[w].skew()
        train_df[f"FEATURE_{source_col}Kurt{w}"] = train_rolling[w].kurt()
        train_df[f"FEATURE_{source_col}Zscore{w}"] = (train_data[source_col] - train_rolling[w].mean()) / train_rolling[w].std()

        test_df[f"FEATURE_{source_col}Skew{w}"] = test_rolling[w].skew()
        test_df[f"FEATURE_{source_col}Kurt{w}"] = test_rolling[w].kurt()
        test_df[f"FEATURE_{source_col}Zscore{w}"] = (test_data[source_col] - test_rolling[w].mean()) / test_rolling[w].std()

rolling_TrainRelativeDiffPct100 = bn.move_rank(train_data["RelativeDiff"].values, window=100)
rolling_TestRelativeDiffPct100 = bn.move_rank(test_data["RelativeDiff"].values, window=100)

train_df[f"FEATURE_RelativeDiffPct100"] = rolling_TrainRelativeDiffPct100 / 100.0
test_df[f"FEATURE_RelativeDiffPct100"] = rolling_TestRelativeDiffPct100 / 100.0

train_df[f"TARGET_UpFlag"] = (train_data["Open"] < train_data["Close"]).shift(-1)
test_df[f"TARGET_UpFlag"] = (test_data["Open"] < test_data["Close"]).shift(-1)

train_df = train_df.dropna()
test_df = test_df.dropna()
train_df["TARGET_UpFlag"] = train_df["TARGET_UpFlag"].astype(int)
test_df["TARGET_UpFlag"] = test_df["TARGET_UpFlag"].astype(int)

XGBoost Training

In [ ]:
# 1. Separate features (X) and target (y)
target_col = "TARGET_UpFlag"
random_state = 64  # Used when fitting XGB Classifier

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# 2. Initialize XGBoost Classifier
model = xgb.XGBClassifier(
    n_estimators=250,           # Trees
    max_depth=5,                # Decision rules depth
    learning_rate=0.02,
    subsample=0.7,
    colsample_bytree=0.6,
    min_child_weight=16,        # Requires at least 16 samples to create a terminal leaf node
    gamma=1.5,
    random_state=random_state,
    eval_metric='logloss',
    n_jobs=-1,
    early_stopping_rounds=15    # Stops fitting when no improvement after consecutive 15 rounds
)

# 3. Train & save the model
# We use the test set as validation to monitor early stopping
print("Training XGBoost model...")
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=10,                  # Print progress every 30 trees
)

os.makedirs("models", exist_ok=True)
model.save_model("models/xgboost_1h_kline.json")

# 4. Evaluate the model
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n=== Evaluation Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 5. Check Feature Importance
importance = pd.Series(model.feature_importances_, index=X_train.columns)
print("\n=== Features Importance ===")
print(importance.nlargest(10))